# DSA 504 — Class 18
## Shell Automation, and a PowerShell Comparison

**Date:** Wednesday, Nov 4
**Today:** HW5 is assigned at the end of class.

---

> **A note on running this notebook:** cells below use `%%bash` to run real shell commands. If a cell fails with something like `bash: command not found` or Jupyter can't locate `bash`, this usually means Git Bash isn't on your system's PATH yet (common on Windows). Fix: find where Git Bash is installed (commonly `C:\Program Files\Git\bin`), add that folder to your System PATH environment variable, then **fully restart** Jupyter (not just the kernel) before trying again. If it still doesn't work, come to office hours rather than losing class time to it.

Classes 15 and 17 covered individual shell commands, typed one at a time. Today covers turning a sequence of commands into a **script** — a saved, reusable file you can run again and again — plus a brief look at **PowerShell**, Windows' own scripting language, for students who'd rather script natively on Windows instead of through Git Bash/WSL.

### Learning goals
By the end of this class, you will be able to:
- Write a shell script with a shebang line, variables, and comments
- Accept a command-line argument in a script
- Use a `for` loop and an `if` conditional in a script
- Make a script executable and run it
- Recognize the PowerShell equivalents of the bash commands you already know


In [5]:
%%bash
cd ~
mkdir -p dsa504_class18
cd dsa504_class18
cat > sales_log.txt << 'EOF'
2026-01-05,Utica,electronics,4521.75
2026-01-05,Albany,groceries,980.10
2026-01-06,Rome,clothing,1102.00
2026-01-06,Utica,toys,340.00
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-08,Utica,groceries,615.40
2026-01-08,Rome,toys,290.00
2026-01-09,Syracuse,clothing,875.60
2026-01-09,Albany,groceries,1020.00
2026-01-10,Utica,electronics,3390.00
2026-01-10,Rome,groceries,740.15
EOF
cat sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-05,Albany,groceries,980.10
2026-01-06,Rome,clothing,1102.00
2026-01-06,Utica,toys,340.00
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-08,Utica,groceries,615.40
2026-01-08,Rome,toys,290.00
2026-01-09,Syracuse,clothing,875.60
2026-01-09,Albany,groceries,1020.00
2026-01-10,Utica,electronics,3390.00
2026-01-10,Rome,groceries,740.15


All cells below `cd ~/dsa504_class18` explicitly at the start — do the same in your own terminal, since (as we found in Class 15) a working directory doesn't necessarily carry over between separate command sessions.


## 1. Your First Shell Script

A script is just a text file containing commands, run top to bottom — the same idea as a Python script from Class 3, in a different language.

**The shebang line** (`#!/bin/bash`) goes at the very top of every bash script. It tells the system which program should run this file — without it, your computer might not know this text file is meant to be executed as bash commands.


In [6]:
%%bash
cd ~/dsa504_class18
cat > hello.sh << 'EOF'
#!/bin/bash
# This is a comment -- lines starting with # are ignored when the script runs
echo "Hello from a shell script"
echo "Today's date is:"
date
EOF
cat hello.sh

#!/bin/bash
# This is a comment -- lines starting with # are ignored when the script runs
echo "Hello from a shell script"
echo "Today's date is:"
date


### Making it executable and running it

A new file isn't runnable by default — you have to explicitly grant it execute permission, the same `chmod` command from Class 15.


In [7]:
%%bash
cd ~/dsa504_class18
chmod +x hello.sh
ls -l hello.sh

-rwxr-xr-x 1 nvasquez 197609 152 Nov  4 09:32 hello.sh


In [8]:
%%bash
cd ~/dsa504_class18
./hello.sh

Hello from a shell script
Today's date is:
Wed Nov  4 09:32:51     2026


**`./hello.sh`** — the `./` is required; just typing `hello.sh` won't work, because (for safety reasons) the shell doesn't search your *current* folder for programs to run unless you tell it to explicitly.


## 2. Variables and Command-Line Arguments

Scripts become genuinely useful once they can take input instead of always doing the exact same thing.


In [9]:
%%bash
cd ~/dsa504_class18
cat > greet.sh << 'EOF'
#!/bin/bash
name="Student"
echo "Hello, $name!"
EOF
chmod +x greet.sh
./greet.sh

Hello, Student!


**Command-line arguments** are values typed after the script name when you run it — `$1` is the first argument, `$2` the second, and so on (the same `$1`/`$2` idea from any command-line tool, including Python's `sys.argv` if you've seen that elsewhere).


In [14]:
%%bash
cd ~/dsa504_class18
cat > category_report.sh << 'EOF'
#!/bin/bash
# category_report.sh - report sales for a given category
# Usage: ./category_report.sh <category>

category=$1

echo "Report for category: $category"
echo "-----------------------------"
grep "$category" sales_log.txt
count=$(grep -c "$category" sales_log.txt)
echo "-----------------------------"
echo "Total matching rows: $count"
EOF
chmod +x category_report.sh
./category_report.sh electronics

Report for category: electronics
-----------------------------
2026-01-05,Utica,electronics,4521.75
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-10,Utica,electronics,3390.00
-----------------------------
Total matching rows: 4


**`$(...)`** runs a command and captures its output into a variable — here, `count=$(grep -c "$category" sales_log.txt)` stores the count so it can be printed later in the script. This is the shell equivalent of assigning a function's return value to a variable back in Class 3.


## 3. Control Flow: `for` Loops and `if` Conditionals

Same concepts as Class 2's Python loops and conditionals — different syntax, same underlying idea.


In [7]:
%%bash
for store in Utica Albany Rome Syracuse; do
  echo "Checking $store..."
done

Checking Utica...
Checking Albany...
Checking Rome...
Checking Syracuse...


In [15]:
%%bash
cd ~/dsa504_class18
count=$(grep -c "electronics" sales_log.txt)
if [ "$count" -gt 3 ]; then
  echo "High volume category: $count transactions"
else
  echo "Low volume category: $count transactions"
fi

High volume category: 4 transactions


**Reading the `if` syntax:** `[ "$count" -gt 3 ]` is bash's way of writing a comparison — `-gt` means "greater than" (bash doesn't use `>` for numeric comparison in this context, since `>` already means redirection, as you learned in Class 15). The quotes around `$count` are a defensive habit — without them, an empty or multi-word value could break the comparison.


## 4. Putting It Together: A Real Automation Script

This is what automation actually looks like — a loop that does the same multi-step job for every store, instead of you typing the same commands four times by hand.


In [16]:
%%bash
cd ~/dsa504_class18
cat > store_summary.sh << 'EOF'
#!/bin/bash
# store_summary.sh - generate a summary report for every store automatically

rm -f store_summary_report.txt

for store in Utica Albany Rome Syracuse; do
  count=$(grep -c "$store" sales_log.txt)
  total=$(grep "$store" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
  echo "$store: $count transactions, total revenue \$$total" >> store_summary_report.txt
done

cat store_summary_report.txt
EOF
chmod +x store_summary.sh
./store_summary.sh

Utica: 4 transactions, total revenue $8867.15
Albany: 3 transactions, total revenue $3875.35
Rome: 3 transactions, total revenue $2132.15
Syracuse: 2 transactions, total revenue $3086.1


**Why this matters:** imagine doing this by hand for 50 stores instead of 4 — this script does it correctly, identically, every time, in under a second. This is the entire point of shell automation: turning a repetitive manual task into something you run once and trust.


## 5. A Brief PowerShell Comparison

**PowerShell** is Windows' own, native scripting language — a real alternative to Git Bash/WSL if you'd rather script directly in Windows' own tools. You are not required to use PowerShell for this course, but it's worth recognizing the equivalents.

| Task | bash | PowerShell |
|---|---|---|
| List files | `ls` | `Get-ChildItem` (alias: `ls` or `dir`) |
| Current directory | `pwd` | `Get-Location` (alias: `pwd`) |
| Change directory | `cd` | `Set-Location` (alias: `cd`) |
| Print text | `echo` | `Write-Output` (alias: `echo`) |
| Set a variable | `name="value"` | `$name = "value"` |
| Comment | `# comment` | `# comment` |
| For loop | `for x in a b c; do ... done` | `foreach ($x in "a","b","c") { ... }` |
| If statement | `if [ "$x" -gt 3 ]; then ... fi` | `if ($x -gt 3) { ... }` |
| Search text | `grep "pattern" file` | `Select-String "pattern" file` |

**Notice PowerShell conveniently provides aliases** (`ls`, `pwd`, `cd`, `echo`) that work the same as their bash names, even though the "real" underlying commands have different, longer names (`Get-ChildItem`, etc.) — this was a deliberate design choice to ease the transition for people coming from Unix-like shells.


---
## Guided Practice

Work through these as actual scripts, saved and run the same way as the examples above.


### Exercise 1 — A simple script with an argument
Write a script called `store_lookup.sh` that takes a store name as its one argument and prints every matching row from `sales_log.txt`, plus a count of how many rows matched.


In [31]:
%%bash
# Exercise 1 — your script here
cd ~/dsa504_class18
cat > store_lookup.sh << 'EOF'
#!/bin/bash
# store_lookup.sh - report sales for a given store argument
# Usage: ./store_lookup.sh <store_name>
store=$1
echo "Report for store: $store"
echo "-----------------------------"
grep "$store" sales_log.txt
count=$(grep -c "$store" sales_log.txt)
echo "-----------------------------"
echo "Total matching rows: $count"
EOF
chmod +x store_lookup.sh
./store_lookup.sh Utica


Report for store: Utica
-----------------------------
2026-01-05,Utica,electronics,4521.75
2026-01-06,Utica,toys,340.00
2026-01-08,Utica,groceries,615.40
2026-01-10,Utica,electronics,3390.00
-----------------------------
Total matching rows: 4


### Exercise 2 — A for loop
Write a script (or a direct command) that loops over the four categories (`electronics`, `groceries`, `clothing`, `toys`) and prints each one's total revenue.


In [30]:
%%bash
# Exercise 2 — your script here
cd ~/dsa504_class18
for category in electronics groceries clothing toys; do
  total=$(grep "$category" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
  echo "Category: $category — Total Revenue: \$$total"
done


Category: electronics — Total Revenue: $11997.5
Category: groceries — Total Revenue: $3355.65
Category: clothing — Total Revenue: $1977.6
Category: toys — Total Revenue: $630


### Exercise 3 — An if conditional
Write a script that checks whether Utica's total revenue exceeds $5000, and prints a message saying whether it does or doesn't.


In [29]:
%%bash
# Exercise 3 — your script here
cd ~/dsa504_class18
cat > utica_check.sh << 'EOF'
#!/bin/bash
total=$(grep "Utica" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
# Use awk to evaluate float comparison safely since bash doesn't do decimals natively
exceeds=$(awk -v t="$total" 'BEGIN {print (t > 5000) ? 1 : 0}')

if [ "$exceeds" -eq 1 ]; then
  echo "Utica's total revenue (\$$total) exceeds \$5000!"
else
  echo "Utica's total revenue (\$$total) does not exceed \$5000."
fi
EOF
chmod +x utica_check.sh
./utica_check.sh


Utica's total revenue ($8867.15) exceeds $5000!


### Exercise 4 (stretch) — Full automation
Write a script that loops over all four categories, and for each one, writes a line to a file called `category_report.txt` showing the category name, transaction count, and total revenue — similar in spirit to `store_summary.sh` above, but by category instead of by store.


In [32]:
%%bash
# Exercise 4 — your script here
cd ~/dsa504_class18
cat > category_summary.sh << 'EOF'
#!/bin/bash
# category_summary.sh - generate a summary report by category automatically
rm -f category_report.txt

for category in electronics groceries clothing toys; do
  count=$(grep -c "$category" sales_log.txt)
  total=$(grep "$category" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
  echo "$category: $count transactions, total revenue \$$total" >> category_report.txt
done

cat category_report.txt
EOF
chmod +x category_summary.sh
./category_summary.sh


electronics: 4 transactions, total revenue $11997.5
groceries: 4 transactions, total revenue $3355.65
clothing: 2 transactions, total revenue $1977.6
toys: 2 transactions, total revenue $630


---
## Solutions

Try each exercise yourself first. These have been tested and confirmed correct.


In [14]:
%%bash
# Solution — Exercise 1
cd ~/dsa504_class18
cat > store_lookup.sh << 'EOF'
#!/bin/bash
store=$1
grep "$store" sales_log.txt
count=$(grep -c "$store" sales_log.txt)
echo "Matching rows: $count"
EOF
chmod +x store_lookup.sh
./store_lookup.sh Rome

2026-01-06,Rome,clothing,1102.00
2026-01-08,Rome,toys,290.00
2026-01-10,Rome,groceries,740.15
Matching rows: 3


In [15]:
%%bash
# Solution — Exercise 2
cd ~/dsa504_class18
for category in electronics groceries clothing toys; do
  total=$(grep "$category" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
  echo "$category: \$$total"
done

electronics: $11997.5
groceries: $3355.65
clothing: $1977.6
toys: $630


In [33]:
%%bash
# Solution — Exercise 3
cd ~/dsa504_class18
grep "Utica" sales_log.txt | awk -F',' '{sum += $4} END {
  if (sum > 5000)
    print "Utica exceeds $5000 (total: $" sum ")"
  else
    print "Utica does not exceed $5000 (total: $" sum ")"
}'

Utica exceeds $5000 (total: $8867.15)


In [17]:
%%bash
# Solution — Exercise 4
cd ~/dsa504_class18
rm -f category_report.txt
for category in electronics groceries clothing toys; do
  count=$(grep -c "$category" sales_log.txt)
  total=$(grep "$category" sales_log.txt | awk -F',' '{sum += $4} END {print sum}')
  echo "$category: $count transactions, total revenue \$$total" >> category_report.txt
done
cat category_report.txt

electronics: 4 transactions, total revenue $11997.5
groceries: 4 transactions, total revenue $3355.65
clothing: 2 transactions, total revenue $1977.6
toys: 2 transactions, total revenue $630


**Note on Exercise 3's solution:** bash's own `if [ ... -gt ... ]` syntax only compares whole numbers, not decimals like `8867.15` — so the comparison is done inside `awk` instead, which handles decimals natively. This also keeps the script portable: a common alternative (the `bc` calculator tool) works on Mac/Linux/WSL but **isn't reliably available in Git Bash on Windows**, so it's avoided here on purpose.


---
## Wrap-up

**Recap:** writing a script with a shebang line and comments; making a script executable with `chmod +x` and running it with `./`; using variables and command-line arguments (`$1`, `$2`); `for` loops and `if` conditionals in bash syntax; a real multi-store automation example; and the PowerShell equivalents of the bash commands you already know.

**Common mistakes to watch for:**
- Forgetting `chmod +x` before trying to run a new script
- Forgetting the `./` prefix when running a script in the current directory
- Missing the shebang line, or putting it somewhere other than the very first line
- Using `>` instead of `-gt` for numeric comparisons inside `if [ ... ]`

**HW5 is assigned today.**

**Before next class (Nov 9):** Class 19 begins R — a second language for data analysis, building directly on the pandas concepts from Classes 4–5, just with new syntax.
